In [ ]:
import torch
import torch.nn as nn

In [ ]:
def conv5x5(in_chan,out_chan):
    return nn.Sequential(
        nn.Conv2d( in_chan,out_chan,kernel_size=3,padding=1),
        # inplace:
        # x -> Module -> y
        # x -> Module -> x   Inplace
        nn.ReLU(inplace=True),
        nn.Conv2d(out_chan,out_chan,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),

        # Reducimos las dimensiones espaciales
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

In [ ]:
def conv7x7(in_chan,out_chan):
      return nn.Sequential(
        nn.Conv2d(in_chan,out_chan,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_chan,out_chan,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_chan,out_chan,kernel_size=3,padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )

In [ ]:
class VGG(nn.Module):
    def __init__(self,n_classes):
        super(VGG,self).__init__()

        self.ConvLayers = nn.Sequential(
            conv5x5(  3, 64),
            conv5x5( 64,128),

            conv7x7(128,256),
            conv7x7(256,512),
            conv7x7(512,512)
        )

        # Fully connect = MLP
        self.FC = nn.Sequential(
            nn.Linear(7*7*512, 4096),
            nn.ReLU(inplace=True),

            # Apagar neuronas de manera aleatoria
            # p: probabilidad de apagar la neurona [0 - 1]
            #    0.5 -> 50% de probabilidad
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, n_classes),
        )

    def forward(self,x):
        #           0       1  2 3
        # x: [n_batch, n_chan, h,w]
        x = self.ConvLayers(x)
        x = torch.flatten(x,1)  # [n_batch, n_cha*h*w]
        x = self.FC(x)
        return x